# AI Impact on Jobs 2030: Workforce Analysis & Forecasting

**Dataset:** AI Impact on Jobs 2030 (Kaggle): 3,000 employee records × 20 columns
**Tool:** Google Colab
**Standard:** ISO/IEC 25059:2023, a quality model for AI systems

### Notebook Flow
1. Import Libraries & Upload Dataset
2. Dataset Overview
3. Data Cleaning
4. Data Visualization
5. Supervised Learning (Regression)
6. Forecasting
7. Unsupervised Learning (K-Means Clustering)
8. ISO/IEC 25059:2023 Model Quality Summary
9. Export Cleaned Dataset (CSV)
10. Conclusion

## 1. Import Libraries & Upload Dataset
We load the libraries for data handling (**pandas, numpy**), charts (**matplotlib, seaborn**) and machine learning (**scikit-learn**).
Then we upload the Kaggle CSV file from our computer into Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")

# =========================
# 1. UPLOAD CSV FILE
# =========================
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"Uploaded file: {filename}")
print(f"Shape: {df.shape}")
display(df.head())

## 2. Dataset Overview
We check the columns, their data types, and the summary statistics.
This tells us which columns are **numeric** (for math and models) and which are **categories** (text labels).

In [ ]:
# =========================
# 2. SHOW COLUMNS AND TYPES
# =========================
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
category_cols = df.select_dtypes(include="object").columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("Category columns:", category_cols)

display(df.describe().T)

## 3. Data Cleaning
Before any analysis, we make sure the data is reliable. *Data quality* is one of the ISO/IEC 25059 requirements.

Steps:
1. Check for **missing values** and fill them (median for numbers, mode for categories)
2. Remove **duplicate rows**
3. Trim extra **spaces** in text columns
4. Drop the **Employee_ID** column, since an ID number does not help prediction
5. Check for **outliers** with the IQR rule
6. **Encode** ordered categories into numbers (e.g. Low=0, Medium=1, High=2) so the models can use them
7. **One-hot encode** `Industry` (one 0/1 column per industry) because industries have no natural order

In [ ]:
# =========================
# 3. DATA CLEANING
# =========================
print(f"Rows before cleaning: {len(df)}")

# 3.1 Missing values
print("\nMissing values per column:")
print(df.isna().sum())

for col in df.columns:
    if df[col].isna().sum() > 0:
        if col in numeric_cols:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

# 3.2 Duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")
df = df.drop_duplicates()

# 3.3 Trim spaces in text columns
for col in category_cols:
    df[col] = df[col].str.strip()

# 3.4 Drop ID column
df = df.drop(columns=["Employee_ID"])

# 3.5 Outlier check (IQR rule)
print("\nOutliers per numeric column (IQR rule):")
for col in df.select_dtypes(include=np.number).columns:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"  {col}: {outliers}")

# 3.6 Encode ordered categories into numbers
df["Education_Num"] = df["Education_Level"].map({"High School": 0, "Bachelor": 1, "Master": 2, "PhD": 3})
df["Automation_Num"] = df["Automation_Level"].map({"Low": 0, "Medium": 1, "High": 2})
df["AI_Tool_Num"] = df["AI_Tool_Usage"].map({"Low": 0, "Moderate": 1, "High": 2})
df["Remote_Num"] = df["Remote_Work_Possibility"].map({"No": 0, "Hybrid": 1, "Yes": 2})
df["Company_Num"] = df["Company_Size"].map({"Startup": 0, "Medium": 1, "Enterprise": 2})
df["Upskilling_Num"] = df["Upskilling_Needed"].map({"No": 0, "Yes": 1})

# 3.7 One-hot encode Industry (no natural order)
industry_dummies = pd.get_dummies(df["Industry"], prefix="Ind", dtype=int)
df = pd.concat([df, industry_dummies], axis=1)
industry_cols = industry_dummies.columns.tolist()

# Risk band (used in charts)
df["Risk_Band"] = pd.cut(df["AI_Replacement_Risk"], bins=[0, 0.30, 0.70, 1.0],
                         labels=["Low", "Medium", "High"], include_lowest=True)

print(f"\nRows after cleaning: {len(df)}")
print(f"Remaining missing values: {df.isna().sum().sum()}")
display(df.head())

## 4. Data Visualization
Charts help us see patterns in the workforce before building models.
- **Bar chart:** average AI replacement risk by job title
- **Pie chart:** share of employees by risk band
- **Histogram:** salary distribution
- **Scatter plot:** AI risk vs. future demand
- **Boxplot:** salary by education level
- **Heatmap:** correlation between numeric columns

In [ ]:
# =========================
# 4.1 BAR CHART
# =========================
job_risk = df.groupby("Job_Title")["AI_Replacement_Risk"].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
job_risk.plot(kind="bar", color="skyblue")
plt.title("Average AI Replacement Risk by Job Title")
plt.xlabel("Job Title")
plt.ylabel("Average AI Replacement Risk")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 4.2 PIE CHART
# =========================
pie_data = df["Risk_Band"].value_counts()

plt.figure(figsize=(7, 7))
pie_data.plot.pie(autopct="%1.1f%%", startangle=90, ylabel="")
plt.title("Employees by AI Risk Band")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 4.3 HISTOGRAM
# =========================
plt.figure(figsize=(8, 5))
sns.histplot(df["Average_Salary_USD"], bins=20, kde=True, color="orange")
plt.title("Histogram of Average Salary (USD)")
plt.xlabel("Average Salary (USD)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 4.4 SCATTER PLOT
# =========================
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="AI_Replacement_Risk", y="Future_Demand_Score", hue="Risk_Band", alpha=0.6)
plt.title("AI Replacement Risk vs Future Demand Score")
plt.xlabel("AI Replacement Risk")
plt.ylabel("Future Demand Score")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 4.5 BOXPLOT
# =========================
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="Education_Level", y="Average_Salary_USD",
            order=["High School", "Bachelor", "Master", "PhD"])
plt.title("Average Salary by Education Level")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 4.6 HEATMAP
# =========================
corr_cols = ["Years_Experience", "AI_Replacement_Risk", "Future_Demand_Score", "Average_Salary_USD",
             "Job_Growth_2030", "Work_Hours_Per_Week", "Performance_Score", "Job_Satisfaction"]

plt.figure(figsize=(10, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 5. Supervised Learning (Regression)
**Supervised learning** trains a model on examples where the answer is already known.
Here the answer is the target column **`Job_Growth_2030`**, the projected % job growth by 2030.

- **Features (X):** AI risk, future demand, automation level, AI tool usage, experience, education, salary, industry
- **Target (y):** Job_Growth_2030
- **Split:** 80% training, 20% testing
- **Models:** Linear Regression (simple, explainable) and Random Forest Regressor (handles non-linear patterns)
- **Metrics:** R² (how much variation the model explains), MAE and RMSE (average error in % points)

In [ ]:
# =========================
# 5. SUPERVISED LEARNING - REGRESSION
# =========================
features = ["AI_Replacement_Risk", "Future_Demand_Score", "Automation_Num", "AI_Tool_Num",
            "Years_Experience", "Education_Num", "Average_Salary_USD"] + industry_cols
target = "Job_Growth_2030"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
}

results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    results.append({
        "Model": name,
        "R2": round(r2_score(y_test, y_pred), 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 3),
        "RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred)), 3)
    })

results_df = pd.DataFrame(results)
display(results_df)

# Linear Regression coefficients (explainability)
lr = models["Linear Regression"]
coef_df = pd.DataFrame({"Feature": features, "Coefficient": lr.coef_}).sort_values("Coefficient")
display(coef_df)

In [ ]:
# =========================
# 5.1 ACTUAL vs PREDICTED
# =========================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, y_pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, y_pred, alpha=0.5)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
    ax.set_title(f"{name}: Actual vs Predicted")
    ax.set_xlabel("Actual Job Growth 2030 (%)")
    ax.set_ylabel("Predicted Job Growth 2030 (%)")
plt.tight_layout()
plt.show()

# Feature importance (Random Forest)
rf = models["Random Forest"]
importance = pd.Series(rf.feature_importances_, index=features).sort_values()

plt.figure(figsize=(8, 5))
importance.plot(kind="barh", color="mediumseagreen")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 6. Forecasting
We use the trained regression model to **forecast job growth from 2026 to 2030** for each industry.

How it works:
1. The model predicts the total 2030 job growth (%) for every employee, then we average it per industry
2. We convert that total into a **yearly growth rate**: `(1 + total%)^(1/4) - 1` (4 years: 2026 → 2030)
3. Starting from a base index of **100 jobs in 2026**, we project each year up to 2030

We also forecast job growth for a **new employee profile** that is not in the dataset.

In [ ]:
# =========================
# 6. FORECASTING (2026 - 2030)
# =========================
best_model = models["Linear Regression"]   # simple and explainable model

df["Predicted_Growth_2030"] = best_model.predict(df[features])
industry_growth = df.groupby("Industry")["Predicted_Growth_2030"].mean().sort_values(ascending=False)

years = [2026, 2027, 2028, 2029, 2030]
forecast = pd.DataFrame(index=years)

for industry, total_growth in industry_growth.items():
    yearly_rate = (1 + total_growth / 100) ** (1 / 4) - 1
    forecast[industry] = [100 * (1 + yearly_rate) ** (year - 2026) for year in years]

print("Forecasted job index per industry (2026 = 100):")
display(forecast.round(1))

plt.figure(figsize=(12, 6))
for industry in forecast.columns:
    plt.plot(forecast.index, forecast[industry], marker="o", label=industry)
plt.title("Job Growth Forecast by Industry (2026 = 100)")
plt.xlabel("Year")
plt.ylabel("Job Index")
plt.xticks(years)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 6.1 FORECAST FOR A NEW EMPLOYEE
# =========================
# Change these values to try a different profile
new_employee = {
    "AI_Replacement_Risk": 0.80,     # 0.0 - 1.0
    "Future_Demand_Score": 0.40,     # 0.0 - 1.0
    "Automation_Num": 2,             # 0 Low, 1 Medium, 2 High
    "AI_Tool_Num": 0,                # 0 Low, 1 Moderate, 2 High
    "Years_Experience": 5,
    "Education_Num": 1,              # 0 High School, 1 Bachelor, 2 Master, 3 PhD
    "Average_Salary_USD": 60000
}
new_industry = "Retail"              # any industry from the dataset

for col in industry_cols:
    new_employee[col] = 1 if col == f"Ind_{new_industry}" else 0
new_employee = pd.DataFrame([new_employee])[features]

for name, model in models.items():
    print(f"{name} forecast ({new_industry}): {model.predict(new_employee)[0]:.2f}% job growth by 2030")

## 7. Unsupervised Learning (K-Means Clustering)
**Unsupervised learning** finds groups in data **without a target column**.
We use **K-Means** to group employees with similar profiles.

1. **Scale** the features so large numbers like salary do not dominate
2. Use the **Elbow Method** to pick the number of clusters (k)
3. Train K-Means and describe each cluster
4. Use **PCA** to reduce the data to 2 dimensions so we can plot the clusters

In [ ]:
# =========================
# 7. UNSUPERVISED LEARNING - K-MEANS
# =========================
cluster_features = ["AI_Replacement_Risk", "Future_Demand_Score", "Average_Salary_USD",
                    "Years_Experience", "Job_Growth_2030", "Automation_Num", "AI_Tool_Num"]

X_scaled = StandardScaler().fit_transform(df[cluster_features])

# 7.1 Elbow Method
inertia = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker="o")
plt.title("Elbow Method for Optimal k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 7.2 TRAIN K-MEANS & VISUALIZE
# =========================
k = 3   # chosen from the elbow chart
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

print("Employees per cluster:")
print(df["Cluster"].value_counts().sort_index())

print("\nAverage profile of each cluster:")
display(df.groupby("Cluster")[cluster_features].mean().round(2))

# PCA to 2D for plotting
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=pca_result[:, 0], y=pca_result[:, 1], hue=df["Cluster"], palette="Set1", alpha=0.6)
plt.title("Employee Clusters (PCA 2D View)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.tight_layout()
plt.show()

## 8. ISO/IEC 25059:2023 Model Quality Summary
**ISO/IEC 25059:2023** is a quality model for AI systems. The table below maps each part of this notebook to the quality characteristics it addresses.

| ISO/IEC 25059 Characteristic | How this notebook addresses it |
|---|---|
| **Functional Correctness** | Models are tested on unseen data (20% test set) using R², MAE and RMSE |
| **Data Quality** | Section 3 checks missing values, duplicates and outliers |
| **Transparency / Interpretability** | Linear Regression coefficients and Random Forest feature importance show *why* the model predicts what it does |
| **Functional Adaptability** | The forecast and the new-employee prediction show the model working on new inputs |
| **Robustness** | Two different algorithms are compared on the same split to check that results agree |

In [ ]:
# =========================
# 8. QUALITY SUMMARY
# =========================
print("=" * 60)
print("        ISO/IEC 25059:2023 MODEL EVALUATION")
print("=" * 60)
print(results_df.to_string(index=False))
print("=" * 60)
print(f"Rows used after cleaning : {len(df)}")
print(f"Missing values remaining : {df[features].isna().sum().sum()}")
print(f"Clusters found (K-Means)  : {k}")

## 9. Export Cleaned Dataset (CSV)
We save our results as CSV files and download them from Colab:
- **`AI_Impact_on_Jobs_2030_cleaned.csv`**: the cleaned dataset with the encoded columns, the model's predicted job growth, and each employee's K-Means cluster
- **`AI_Impact_on_Jobs_2030_forecast.csv`**: the 2026–2030 job growth forecast per industry

In [ ]:
# =========================
# 9. EXPORT CSV
# =========================
cleaned_file = "AI_Impact_on_Jobs_2030_cleaned.csv"
forecast_file = "AI_Impact_on_Jobs_2030_forecast.csv"

df.to_csv(cleaned_file, index=False)
forecast.round(2).to_csv(forecast_file, index_label="Year")

print(f"Saved {cleaned_file}: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Saved {forecast_file}: {forecast.shape[0]} years x {forecast.shape[1]} industries")

files.download(cleaned_file)
files.download(forecast_file)

## 10. Conclusion
- **Data Cleaning:** checked missing values, duplicates and outliers, and encoded the categories into numbers.
- **Visualization:** showed which jobs face the highest AI replacement risk and how salary, risk and demand are spread across employees.
- **Supervised Learning:** Linear Regression and Random Forest predicted Job Growth 2030, and both were scored with R², MAE and RMSE.
- **Forecasting:** projected job growth per industry from 2026 to 2030 and for a new employee profile.
- **Unsupervised Learning:** K-Means split employees into 3 groups: a **high-automation** group, a **low job-growth** group (about +4%) and a **high job-growth** group (about +32%).
- **ISO/IEC 25059:2023:** every model step is mapped to an AI quality characteristic.

> *Note:* This Kaggle dataset has very weak correlations between columns (see the heatmap), so the R² scores are close to 0. The models predict near the average, and their error is about ±15 percentage points. We report this openly, because ISO/IEC 25059 values transparency over inflated results.